In [ ]:
__author__ = "Kay Töpfer, Matthias Groh"
__organization__="SE G QPI QM P"
__copyright__ = "Copyright 2020, Siemens Gas and Power GmbH & Co. KG"
__email__ = "kay.toepfer@siemens.com, groh.matthias@siemens.com"

In [ ]:
import numpy as np
import pandas as pd
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [ ]:
def get_converted_data(data,xData,yData,trainPercShare,randomShuffle,scaling):
    data=data.sample(frac=1, random_state=1) #Shuffle data
    if randomShuffle==True:
        data=data.sample(frac=1) #Shuffle data random
    trainShare=round(len(data)*trainPercShare/100) #Calculate Number of Samples for trainings data
    #Define trainings and test data
    trainData=data[:trainShare]
    testData=data[trainShare:]
    xTrain=trainData.loc[:,xData].values
    yTrain=trainData[yData].values
    xTest=testData.loc[:,xData].values
    yTest=testData[yData].values
    #Scale data
    scaler=None
    if scaling==True:
        try:
            scaler=StandardScaler()
            scaler.fit(xTrain)
            xTrain=scaler.transform(xTrain)
        except:
            scaler.fit(xTest)
            xTest=scaler.transform(xTest)
        try:
            xTest=scaler.transform(xTest)
        except: 
            pass
    return(xTrain,yTrain,xTest,yTest,scaler)

def get_hidden_layer_structure(hiddenLayers,neurons,exactHiddenLayerStructure):
    if exactHiddenLayerStructure==False:
        exactHiddenLayerStructure=[] #Create List for Hidden Layer structure
        while hiddenLayers!=0: #Appends for each Hidden Layer the number of Neurons
            exactHiddenLayerStructure.append(neurons) 
            hiddenLayers-=1
    return(tuple(exactHiddenLayerStructure)) #Convert List into Tuple

def update_result_2outputs(input_,output1,output2,resultEx):
    if input_ in resultEx[resultEx.columns[0]]:
        resultEx=resultEx.drop([input_])
    newRow=pd.Series([input_,output1,output2],index=resultEx.columns)
    resultEx=pd.concat([pd.DataFrame([newRow],index=[input_]),resultEx])
    return(resultEx)

def update_results_4inputs(input1,input2,input3,input4,output,resultEx):
    newRow=pd.Series([input1,input2,input3,input4,output],index=resultEx.columns)
    resultEx=pd.concat([pd.DataFrame([newRow]),resultEx])
    return(resultEx)

def create_decision_tree(data,xData,yData,maxDepth,trainPercShare,randomShuffle,randomState,toPrint):
    convertedData = get_converted_data(data,xData,yData,trainPercShare,randomShuffle,False)
    xTrain=convertedData[0]
    yTrain=convertedData[1]
    xTest=convertedData[2]
    yTest=convertedData[3]
    #Calculate Decision Tree
    clf=DecisionTreeClassifier(criterion='entropy',max_depth=maxDepth,random_state=randomState).fit(xTrain,yTrain)
    #Calculate scores
    testScore=clf.score(xTest,yTest)
    trainScore=clf.score(xTrain,yTrain)
    #Print scores
    if any('est' in i for i in toPrint):
        print('Test Accuracy: '+str(round(testScore*100,1))+'%')
    if any('rain' in i for i in toPrint):
        print('Train Accuracy: '+str(round(trainScore*100,1))+'%')
    return(testScore,trainScore,clf,[xTrain,yTrain,xTest,yTest])

def create_random_forest(data,xData,yData,maxDepth,estimators,trainPercShare,randomShuffle,randomState,toPrint):
    convertedData = get_converted_data(data,xData,yData,trainPercShare,randomShuffle,False)
    xTrain=convertedData[0]
    yTrain=convertedData[1]
    xTest=convertedData[2]
    yTest=convertedData[3]
    #Calculate Decision Tree
    clf=RandomForestClassifier(criterion='entropy',n_estimators=estimators,max_depth=maxDepth,random_state=randomState).fit(xTrain,yTrain)
    #Calculate scores
    testScore=clf.score(xTest,yTest)
    trainScore=clf.score(xTrain,yTrain)
    #Print scores
    if any('est' in i for i in toPrint):
        print('Test Accuracy: '+str(round(testScore*100,1))+'%')
    if any('rain' in i for i in toPrint):
        print('Train Accuracy: '+str(round(trainScore*100,1))+'%')
    return(testScore,trainScore,clf,[xTrain,yTrain,xTest,yTest])

def create_neural_network(data,xData,yData,learningRate,hiddenLayers,neurons,activationFkt,solvingAlgo,maxEpochs,trainPercShare,randomShuffle,randomState,exactHiddenLayerStructure,toPrint):
    #Get converted data
    convertedData = get_converted_data(data,xData,yData,trainPercShare,randomShuffle,True)
    xTrain=convertedData[0]
    yTrain=convertedData[1]
    xTest=convertedData[2]
    yTest=convertedData[3]
    #Get Hidden Layer structure
    hiddenLayerStructure=get_hidden_layer_structure(hiddenLayers,neurons,exactHiddenLayerStructure)
    #Calculate Decision Tree
    clf=MLPClassifier(hidden_layer_sizes=hiddenLayerStructure,activation=activationFkt,solver=solvingAlgo,learning_rate_init=learningRate,max_iter=maxEpochs,random_state=randomState).fit(xTrain,yTrain)
    #Calculate scores
    testScore=clf.score(xTest,yTest)
    trainScore=clf.score(xTrain,yTrain)
    if any('est' in i for i in toPrint):
        print('Test Accuracy: '+str(round(testScore*100,1))+'%')
    if any('rain' in i for i in toPrint):
        print('Train Accuracy: '+str(round(trainScore*100,1))+'%')
    return(testScore,trainScore,clf,[xTrain,yTrain,xTest,yTest])

def create_plot(resultEx,logarithmic):
    style='-'
    if len(resultEx)==1:
        style='o'
    resultEx=resultEx.sort_values(by=[resultEx.columns[0]])
    if logarithmic==True:
        resultEx[resultEx.columns[0]]=np.log10(resultEx[resultEx.columns[0]].astype('float'))
    plt.plot(resultEx[resultEx.columns[0]],resultEx[resultEx.columns[1]],style,label=resultEx.columns[1])
    plt.plot(resultEx[resultEx.columns[0]],resultEx[resultEx.columns[2]],style,label=resultEx.columns[2])
    plt.grid(True)
    plt.ylim(0, 105)
    plt.xlabel(resultEx.columns[0])
    plt.ylabel('Accuracy [%]')
    if logarithmic==True:
        resultEx[resultEx.columns[0]]=np.power(10,resultEx[resultEx.columns[0]])
        plt.xlabel(resultEx.columns[0]+' [*10^x]')
    plt.legend()
    plt.show()

def create_plot_3D(resultEx,activationFkt):
    resultEx=resultEx[resultEx['Activation']==activationFkt]
    #Prepare data
    LearningRate=np.log10(resultEx['Learning Rate'].astype('float'))
    HiddenLayer=resultEx['Hidden Layers']
    Neurons=resultEx['Neurons']
    Score=resultEx['Test Accuracy']
    testii=list(resultEx['Learning Rate'])
    #Plot data
    fig=go.Figure(data=[go.Scatter3d(x=LearningRate,y=HiddenLayer,z=Neurons,mode='markers',marker=dict(size=10,color=Score,colorscale='Viridis',opacity=0.8), hovertemplate ='<br><b>Learning Rate</b>: 10^%{x}'+'<br><b>Number of Hidden Layers</b>: %{y}'+'<br><b>Number of Neurons</b>: %{z}<br>'+'<br><b>Test Accuracy</b>: %{marker.color:.1f}%'+'<extra></extra>')])
    fig.update_layout(scene = dict(xaxis_title='Learning Rate [*10^x]',yaxis_title='Number of Hidden Layers',zaxis_title='Number of Neurons'),width=700,margin=dict(r=20, b=10, l=10, t=10))
    fig.show()

def update_NN(model,data,xData,yData,trainPercShare,randomShuffle,updateData,trainPercShareUpdate,randomShuffleUpdate,iterations,toPrint):
    convertedData = get_converted_data(data,xData,yData,trainPercShare,randomShuffle,True)
    convertedUpdateData = get_converted_data(updateData,xData,yData,trainPercShareUpdate,randomShuffleUpdate,True)
    testScoreBeforeUpdate=model.score(convertedData[2],convertedData[3])
    trainScoreBeforeUpdate=model.score(convertedData[0],convertedData[1])
    for i in range(0,iterations):
        model=model.partial_fit(convertedUpdateData[0],convertedUpdateData[1])
    xTrainUpdated=np.concatenate((convertedData[0],convertedUpdateData[0]))
    yTrainUpdated=np.concatenate((convertedData[1],convertedUpdateData[1]))
    xTestUpdated=np.concatenate((convertedData[2],convertedUpdateData[2]))
    yTestUpdated=np.concatenate((convertedData[3],convertedUpdateData[3]))
    columns_=xData.copy()
    columns_.append(yData)
    UpdatedData=pd.DataFrame(data=np.column_stack((np.concatenate((xTrainUpdated,xTestUpdated)),np.concatenate((yTrainUpdated,yTestUpdated)))),columns=columns_)
    testScoreAfterUpdate=model.score(xTestUpdated,yTestUpdated)
    trainScoreAfterUpdate=model.score(xTrainUpdated,yTrainUpdated)
    if any('est' in i for i in toPrint):
        print('Test accuracy before updating the model: '+str(round(testScoreBeforeUpdate*100,1))+'%')
    if any('rain' in i for i in toPrint):
        print('Train accuracy before updating the model: '+str(round(trainScoreBeforeUpdate*100,1))+'%')
    print()
    if any('est' in i for i in toPrint):
        print('Test accuracy after updating the model:  '+str(round(testScoreAfterUpdate*100,1))+'%')
    if any('rain' in i for i in toPrint):
        print('Train accuracy after updating the model: '+str(round(trainScoreAfterUpdate*100,1))+'%')
    return(model,UpdatedData)

def delete_value(column, value, df):
    df=df[df[column] != value]
    return(df)

In [ ]:
resultEx21=pd.DataFrame(columns=['Max Depth','Decision Tree Test Accuracy','Random Forest Test Accuracy'])
resultEx22=pd.DataFrame(columns=['Number of Estimators','Test Accuracy','Train Accuracy'])
resultEx3=pd.DataFrame(columns=['Learning Rate','Hidden Layers','Neurons','Activation','Test Accuracy'])

## Exercise 2.1: Decision Trees vs. Random Forest

In [ ]:
###################################################### Specify data ##########################################################################

data=pd.read_excel('Numbers-700.xlsx')
xData=[str(i) for i in list(range(1,785))] #Generates a list ['1', '2', '3', ..., '783', '784'], these are the column names
yData='Number'

In [ ]:
####################################################### For profis ###########################################################################

trainPercShare=90 #Number between 0 and 100
randomShuffle=False #True or False
randomState=1 #None or Number

estimatorsDefault=100 #Number

In [ ]:
#################################################### Specify parameters ######################################################################

maxDepth=1 #Number

##############################################################################################################################################

#Create Decision Tree & Random Forest
print('Decision Tree:')
print('')
decisionTree=create_decision_tree(data,xData,yData,maxDepth,trainPercShare,randomShuffle,randomState,['test'])
print('')
print('Random Forest:')
print('')
randomForest=create_random_forest(data,xData,yData,maxDepth,estimatorsDefault,trainPercShare,randomShuffle,randomState,['test'])
print('')
#Save results
resultEx21=update_result_2outputs(maxDepth,decisionTree[0]*100,randomForest[0]*100,resultEx21)
#Plot results
logarithmic=False
create_plot(resultEx21,logarithmic)

## Exercise 2.2: Random Forest
### Explore the Parameter **Number of Estimators** with **Numbers Recognition** Dataset

In [ ]:
####################################################### For profis ###########################################################################

trainPercShare=90 #Number between 0 and 100
randomShuffle=False #True or False
randomState=1 #None or Number

In [ ]:
#################################################### Specify parameters ######################################################################

maxDepth=1 #Number
estimators=1 #Number

##############################################################################################################################################

#Create Random Forest
randomForest=create_random_forest(data,xData,yData,maxDepth,estimators,trainPercShare,randomShuffle,randomState,['test','train'])
#Save results
resultEx22=update_result_2outputs(estimators,randomForest[0]*100,randomForest[1]*100,resultEx22)
#Plot results
logarithmic=False
create_plot(resultEx22,logarithmic)

## Exercise 3: Neural Networks 
### Explore the Parameters **Learning Rate, Hidden Layers, Neurons and Activation Function** with **Numbers Recognition** Dataset

In [ ]:
####################################################### For profis ###########################################################################

solvingAlgo='sgd' #‘lbfgs’, ‘sgd’, ‘adam’
exactHiddenLayerStructure=False #False (same NumNeurons in each HidLay) or from type [100,10,40] ([NumNeur in HidLay1, NumNeur in HidLay2, ... ])
maxEpochs=200 #Number (max number of iterations)
trainPercShare=90 #Number between 0 and 100
randomShuffle=False #True or False
randomState=1 #None or Number

In [ ]:
#################################################### Specify parameters ######################################################################

learningRate=0.0001 #Number
hiddenLayers=10 #Number
neurons=10 #Number
activationFkt='logistic' #'logistic' or 'relu'

##############################################################################################################################################

#Create Neural Network
neuralNetwork=create_neural_network(data,xData,yData,learningRate,hiddenLayers,neurons,activationFkt,solvingAlgo,maxEpochs,trainPercShare,randomShuffle,randomState,exactHiddenLayerStructure,['test'])
#Save results
resultEx3=update_results_4inputs(learningRate,hiddenLayers,neurons,activationFkt,neuralNetwork[0]*100,resultEx3)

### Result for logistic

In [ ]:
create_plot_3D(resultEx3,'logistic')

### Result for relu

In [ ]:
create_plot_3D(resultEx3,'relu')

## Updating an existing model with new data

In [ ]:
###################################################### Specify data ##########################################################################

updateData=pd.read_excel('Numbers-Update.xlsx')

In [ ]:
####################################################### For profis ###########################################################################

solvingAlgo='sgd' #‘lbfgs’, ‘sgd’, ‘adam’
exactHiddenLayerStructure=False #False (same NumNeurons in each HidLay) or from type [100,10,40] ([NumNeur in HidLay1, NumNeur in HidLay2, ... ])
maxEpochs=200 #Number (max number of iterations)
trainPercShare=90 #Number between 0 and 100
randomShuffle=False #True or False
randomState=1 #None or Number

trainPercShareUpdate=100 #Number between 0 and 100
randomShuffleUpdate=False #True or False

In [ ]:
#################################################### Specify parameters ######################################################################

iterations=5

learningRate=0.01 #Number
hiddenLayers=1 #Number
neurons=50 #Number
activationFkt='logistic' #‘identity’, ‘logistic’, ‘tanh’ or ‘relu’

##############################################################################################################################################

#Create Neural Network
neuralNetwork=create_neural_network(data,xData,yData,learningRate,hiddenLayers,neurons,activationFkt,solvingAlgo,maxEpochs,trainPercShare,randomShuffle,randomState,exactHiddenLayerStructure,[])
#Update Neural Network
[updatedNeuralNetwork,updatedData]=update_NN(neuralNetwork[2],data,xData,yData,trainPercShare,randomShuffle,updateData,trainPercShareUpdate,randomShuffleUpdate,iterations,['test'])

## For more details go to: https://scikit-learn.org/stable/